# Generating Synthetic Data for BVSIMC

Creates a binary matrix from low-rank side information (`U`, `V`), with the first `k` columns carrying signal and the rest pure noise. Positive entries can be randomly flipped to zero to simulate missing/mislabeled labels.

In [1]:
import numpy as np
from sklearn.utils import check_random_state


def make_binary_imc_data(n1, d1, n2, d2, k, scale=0.05, noise=0.0,
                          target_density=0.1, random_state=None):
    """Simulate a binary inductive-matrix-completion problem.

    Y = sigmoid-free binary threshold of U @ A @ (V @ B).T, where only the
    first k columns of U, V carry signal (A, B are identity projections).
    `noise` randomly flips a fraction of the positive entries to 0.

    Returns U, V, Y (clean), Y_noisy (some positives flipped to 0).
    """
    rng = check_random_state(random_state)
    assert d1 >= k and d2 >= k

    U = rng.normal(scale=scale, size=(n1, d1))
    V = rng.normal(scale=scale, size=(n2, d2))
    A, B = np.eye(d1, k), np.eye(d2, k)

    Y_cont = (U @ A) @ (V @ B).T
    threshold = np.quantile(Y_cont, 1.0 - target_density)
    Y = (Y_cont >= threshold).astype(float)

    Y_noisy = Y.copy()
    if noise > 0:
        pos = Y_cont >= threshold
        perturbed = Y_cont[pos] + rng.normal(scale=noise, size=int(pos.sum()))
        Y_noisy[pos] = (perturbed >= threshold).astype(float)

    return U, V, Y, Y_noisy


## Generate one example dataset

400 x 600 matrix, 25 informative features out of 100, ~10% positive density.

In [2]:
U, V, Y, Y_noisy = make_binary_imc_data(
    n1=400, d1=100, n2=600, d2=100, k=25,
    scale=0.05, noise=0.02, target_density=0.1,
    random_state=0,
)

print(f"U: {U.shape}, V: {V.shape}, Y: {Y.shape}")
print(f"Positive rate (clean):  {Y.mean():.3f}")
print(f"Positive rate (noisy):  {Y_noisy.mean():.3f}")
print(f"Positives flipped to 0: {int((Y - Y_noisy).sum())}")


U: (400, 100), V: (600, 100), Y: (400, 600)
Positive rate (clean):  0.100
Positive rate (noisy):  0.062
Positives flipped to 0: 9153


## Save for reuse in the BVSIMC demo notebook

In [3]:
import os, gzip, pickle

os.makedirs("data", exist_ok=True)
with gzip.open("data/demo_dataset.gz", "wb") as f:
    pickle.dump({"U": U, "V": V, "Y": Y, "Y_noisy": Y_noisy}, f)

print("Saved to data/demo_dataset.gz")


Saved to data/demo_dataset.gz
